In [1]:
from read_training_data import get_tension_data, get_raw_data, get_lengths, calculate_res_freq
import pandas as pd
import numpy as np


In [2]:
raw_data_path = "training-data/raw/APA_5947a8b0-5152-11ed-84c6-9170f34dd096b_apa4_training"
processed_data_path = "training-data/processed/5947a8b0-5152-11ed-84c6-9170f34dd096b_apa4_training.json"

In [3]:
proc_scans = get_tension_data(processed_data_path, ["U", "X"])


In [4]:
keys_to_remove = {"scanEndMode", "measuredBy", "stage", "apaUuid", "type"}
raw_scans = get_raw_data(raw_data_path, keys_to_remove)

In [5]:
proc_df = pd.DataFrame(proc_scans)
raw_df = pd.DataFrame(raw_scans)
# raw_df.head()

In [6]:
proc_df.head()

,uid,stage,tension
0,UA34,Winding,5.573973
1,UA36,Winding,5.408697
2,UA38,Winding,8.020985
3,UA40,Winding,5.693777
4,UA41,Winding,5.326113


In [7]:
comb_df = pd.merge(raw_df, proc_df, on="uid", how="outer")
comb_df.head(3)

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension
0,UA1002,1002,0,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3870.976801, 3870.854905, 3875.409452, 3883.0...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.1644
1,UA1004,1004,1,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[1328.26138, 1327.855029, 1329.099351, 1332.22...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.331951
2,UA1006,1006,2,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3779.794773, 3780.85742, 3783.779545, 3791.47...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.381051


In [8]:
final_comb_df = comb_df.dropna(subset=['tension'])
numeric_tension = pd.to_numeric(final_comb_df['tension'], errors='coerce') # removes tensions that aren't found
final_comb_df['tension'] = numeric_tension
final_comb_df.shape

(698, 11)

In [9]:
u_side_lengths = get_lengths("training-data/tensions/U_Layer_Tension_Template.xlsx", "U")
x_side_lengths = get_lengths("training-data/tensions/X_Layer_Tension_Template.xlsx", "X")

In [10]:
print(x_side_lengths)

1273.0


In [11]:
u_side_lengths.columns

Index(['length', 'uid'], dtype='str')

In [12]:
u_side_lengths["length"].values

array([69.86160506, 79.70761336, 89.55523435, ..., 70.95958991,
       61.1647848 , 51.39186122], shape=(2282,))

In [13]:
final_df = pd.merge(final_comb_df, u_side_lengths, on="uid", how="inner") # this only includes U layer wires


In [14]:
x_wires = final_comb_df[final_comb_df["uid"].str.startswith("X", na=False)]
x_wires["length"] = x_side_lengths
x_wires.head()

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension,length
648,XA18,18,7,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[1529.835389, 1529.242702, 1531.671658, 1535.8...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.004333,1273.0
649,XA20,20,6,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[4554.047346, 4555.735923, 4561.981086, 4571.3...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.386445,1273.0
650,XA22,22,5,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[1552.343604, 1553.119515, 1555.380363, 1557.3...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.444849,1273.0
651,XA225,225,7,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[933.470875, 933.853257, 934.964307, 937.38942...",X,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.022262,1273.0
652,XA227,227,6,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[4425.846927, 4426.155762, 4432.554884, 4443.6...",X,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.483456,1273.0


In [15]:
final_df = pd.concat([final_df, x_wires], ignore_index=True) # reincludes X layers
final_df.shape

(698, 12)

In [16]:
print(type(final_df['length'].values))
print(type(final_df['tension'].values))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [17]:
# Create a mask that checks if each item is a string
# string_mask = final_df['tension'].apply(lambda x: isinstance(x, str))


numeric_tension = pd.to_numeric(final_df['tension'], errors='coerce') # converts tension not found to NaN
final_df['tension'] = numeric_tension
final_df = final_df.dropna(subset=['tension']) # drops rows with NaN tension values
final_df.shape

(611, 12)

In [18]:
tension = final_df['tension'].values
lengths = final_df['length'].values / 1000 # in meters

rho = 1.52e-4
 

res_freq = calculate_res_freq(tension, lengths, rho) # in Hz

res_freq.shape

final_df['res_freq'] = res_freq

In [19]:
final_df[final_df["uid"].str.startswith("X", na=False)].head(3)

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension,length,res_freq
618,XA18,18,7,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[1529.835389, 1529.242702, 1531.671658, 1535.8...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.004333,1273.0,78.064251
619,XA20,20,6,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[4554.047346, 4555.735923, 4561.981086, 4571.3...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.386445,1273.0,80.509918
620,XA22,22,5,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[1552.343604, 1553.119515, 1555.380363, 1557.3...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.444849,1273.0,80.877216


In [ ]:
# validating the resonant frequency calculation
final_df[(final_df['res_freq'] < 50)].sort_values(by='res_freq', ascending=False)[["parent_folder", "uid", "res_freq"]].head(5)

,parent_folder,uid,res_freq
28,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,UA143,48.837003
243,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,UA943,46.465810
131,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,UA543,46.429169


In [27]:
x=final_df[final_df["uid"].str.startswith("X", na=False)].iloc[1]
print(max(x['freq']))

112.00000028
